This notebook is used to complete the missing values from `_man` to `_auto`

In [1]:
import os
import sys
from dotenv import load_dotenv

# Get the project root directory (two levels up from this notebook)
project_root = "/home/user/Desktop/AI Code/AIoT_KHH_Airport_AI_Agent/"
print(project_root)

# Change working directory to project root so relative paths work
os.chdir(project_root)
print(f"Changed working directory to: {os.getcwd()}")

# Add project root to Python path so we can import from 'agents' package
if project_root not in sys.path:
    sys.path.append(project_root)

env_path = os.path.join(project_root, ".env")
load_dotenv(env_path)

# Verify the database path from .env
SQL_DATABASE_PATH = os.getenv("SQL_DATABASE_PATH")
print(f"SQL_DATABASE_PATH from .env: {SQL_DATABASE_PATH}")
print(f"Database file exists: {os.path.exists(SQL_DATABASE_PATH)}")
SQL_DATABASE_PATH

/home/user/Desktop/AI Code/AIoT_KHH_Airport_AI_Agent/
Changed working directory to: /home/user/Desktop/AI Code/AIoT_KHH_Airport_AI_Agent
SQL_DATABASE_PATH from .env: database/airport_data.db
Database file exists: True


'database/airport_data.db'

In [2]:
# Step 1: Create a backup of the database (IMPORTANT!)
import shutil
from datetime import datetime
import sqlite3
import pandas as pd

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
backup_db = SQL_DATABASE_PATH.replace('.db', f'_backup_{timestamp}.db')

shutil.copy2(SQL_DATABASE_PATH, backup_db)
print(f"✅ Backup created: {backup_db}")
print(f"Original DB size: {os.path.getsize(SQL_DATABASE_PATH) / (1024*1024):.2f} MB")


✅ Backup created: database/airport_data_backup_20251016_111853.db
Original DB size: 7.34 MB


## Arrival

In [145]:
# Step 2: Load data using pandas
# Load from SQLite (current state - arrival_auto)
conn = sqlite3.connect(SQL_DATABASE_PATH)
arrival_sql = pd.read_sql("SELECT * FROM arrivals", conn)

# Load from CSV (source of truth - arrival_man)
arrival_man_df = pd.read_csv("database/arrival_man_bk.csv")

print(f"Arrival Auto (SQLite) rows: {len(arrival_sql)}")
print(f"Arrival Man (CSV) rows: {len(arrival_man_df)}")
print(f"\nArrival Auto columns: {list(arrival_sql.columns)}")
print(f"Arrival Man columns (first 20): {list(arrival_man_df.columns[:20])}")




Arrival Auto (SQLite) rows: 29083
Arrival Man (CSV) rows: 31111

Arrival Auto columns: ['FID', 'AircraftType', 'AirlineIATA', 'ArrivalDateTime', 'Bay', 'Cancel', 'Cargo', 'DepartureAirportIATA', 'FDate', 'FlightNumber', 'LoadCapacity', 'Passanger', 'RWYARR', 'Reg', 'SCHE_TYPE', 'SeatCapacity', 'amhsATA', 'logtime']
Arrival Man columns (first 20): ['FID', 'Airline', 'FlightNumber', 'Bay', 'STA', 'arrtime', 'statusarr', 'notearr', 'respon', 'full', 'Passanger', 'Infant', 'transit', 'diplomat', 'other', 'note', 'payable', 'Bag', 'Post', 'Cargo']


In [146]:
arrival_auto_df = arrival_sql.copy()

In [147]:
# Step 3: Prepare data for matching using composite key

# The matching keys are:
# - AirlineIATA (auto) <--> Airline (man)
# - Bay (auto) <--> Bay (man)
# - FlightNumber (auto) <--> FlightNumber (man)
# - FDate (auto) <--> STA (%Y-%m-%d only) (man)
# - Reg (auto) <--> Reg (man)
# - amhsATA (%Y-%m-%d-h) <--> arrtime (%Y-%m-%d-%h) (man)

# Prepare arrival_man_df

# Rename Airline to AirlineIATA in man data for easier merging
arrival_man_df_prepared = arrival_man_df.copy()
arrival_man_df_prepared['AirlineIATA'] = arrival_man_df_prepared['Airline']

# Convert STA to date format to match FDate
arrival_man_df_prepared['STA_date'] = pd.to_datetime(arrival_man_df['STA'], errors='coerce').dt.date
arrival_auto_df['FDate_date'] = pd.to_datetime(arrival_auto_df['FDate'], errors='coerce').dt.date

# int the BAY column for both of the dataframe
arrival_man_df_prepared['Bay'] = arrival_man_df['Bay'].fillna(0).astype(int)
arrival_auto_df['Bay'] = arrival_auto_df['Bay'].fillna(0).astype(int)

# int the FlightNumber
arrival_man_df_prepared['FlightNumber'] = pd.to_numeric(arrival_man_df['FlightNumber'], errors='coerce')
arrival_man_df_prepared = arrival_man_df_prepared.dropna(subset=['FlightNumber'])  # Drop rows where FlightNumber is NaN
arrival_man_df_prepared['FlightNumber'] = arrival_man_df_prepared['FlightNumber'].astype(int)

arrival_auto_df['FlightNumber'] = pd.to_numeric(arrival_auto_df['FlightNumber'], errors='coerce')
arrival_auto_df = arrival_auto_df.dropna(subset=['FlightNumber'])
arrival_auto_df['FlightNumber'] = arrival_auto_df['FlightNumber'].astype(int)

# actual arrival time
# amhsATA (%Y-%m-%d-h) <--> arrtime (%Y-%m-%d-%h) (man)
arrival_man_df_prepared['actual_arrival_time'] = pd.to_datetime(arrival_man_df['arrtime'], errors='coerce').dt.floor('h')
arrival_auto_df['actual_arrival_time'] = pd.to_datetime(arrival_auto_df['amhsATA'], errors='coerce').dt.floor('h')


In [148]:
def deduplicate_manual_records(df):
    """Remove duplicate manual records, keeping only the first instance for each truly identical record"""
    
    # Create a composite key using ALL data fields (except FID which is unique)
    # Convert all columns to string and concatenate
    df_copy = df.copy()
    
    # Get all columns except FID
    columns_to_compare = [col for col in df_copy.columns if col != 'FID']
    
    # Create composite key from all fields
    df_copy['composite_key'] = df_copy[columns_to_compare].astype(str).agg('_'.join, axis=1)
    
    # Keep only the first occurrence of each truly identical record
    deduplicated = df_copy.drop_duplicates(subset=['composite_key'], keep='first')
    
    # Remove the temporary composite key column
    deduplicated = deduplicated.drop('composite_key', axis=1)
    
    return deduplicated

# Apply deduplication
arrival_man_df_prepared_deduplicated = deduplicate_manual_records(arrival_man_df_prepared)

print(f"Original manual records: {len(arrival_man_df_prepared)}")
print(f"After deduplication: {len(arrival_man_df_prepared_deduplicated)}")
print(f"Removed {len(arrival_man_df_prepared) - len(arrival_man_df_prepared_deduplicated)} truly identical records")

Original manual records: 30890
After deduplication: 29003
Removed 1887 truly identical records


In [93]:
from tqdm import tqdm

# Step 3.5: Test the composite key matching
# Iterate through the arrival_man to check each man
no_match = []

# The `man["FID"]` that has more than one `auto["FID"]`
more_than_one_match = []

# The `man["FID"]` that has exact match
exact_match = []

exact_auto_match_set = set()

# The `auto["FID"]` that has multiple `man["FID"]`
auto_one_to_many_fid = []


for i in tqdm(range(len(arrival_man_df_prepared_deduplicated))):
    man_record = arrival_man_df_prepared_deduplicated.iloc[i]
    
    # man --> auto
    auto_match = arrival_auto_df[
        # (arrival_auto_df['AirlineIATA'] == man_record['AirlineIATA']) &
        # (arrival_auto_df['FlightNumber'] == man_record['FlightNumber']) &
        # (arrival_auto_df['Bay'] == man_record['Bay']) &
        (arrival_auto_df['FDate_date'] == man_record['STA_date']) &
        (arrival_auto_df["Reg"] == man_record["Reg"]) &
        (abs(arrival_auto_df["actual_arrival_time"] - man_record["actual_arrival_time"]) <= pd.Timedelta(hours=1))
    ]
    if not auto_match.empty:
        # Only update if len(match) == 1
        if len(auto_match) == 1:
            exact_match.append(man_record["FID"])
            if auto_match.iloc[0]["FID"] in exact_auto_match_set:
                auto_one_to_many_fid.append(auto_match.iloc[0]["FID"])
            else:
                exact_auto_match_set.add(auto_match.iloc[0]["FID"])

            # Check if this match has already been matched by other man_record

        else: # len(match) is not 1, needs to be inspected
            more_than_one_match.append(man_record["FID"])
    else: # No match len(match) == 0
        no_match.append(man_record["FID"])

len(no_match), len(more_than_one_match), len(exact_match), len(auto_one_to_many_fid)

100%|██████████| 29003/29003 [01:50<00:00, 262.54it/s]


(7689, 16, 21298, 29)

In [96]:
auto_one_to_many_fid[:5]

[125796, 123496, 123212, 121329, 121092]

In [ ]:
man_record = arrival_man_df_prepared_deduplicated[arrival_man_df_prepared_deduplicated["FID"]==105566].iloc[0]
arrival_auto_df[
        # (arrival_auto_df['AirlineIATA'] == man_record['AirlineIATA']) &
        # (arrival_auto_df['FlightNumber'] == man_record['FlightNumber']) &
        # (arrival_auto_df['Bay'] == man_record['Bay']) &
        (arrival_auto_df['FDate_date'] == man_record['STA_date']) &
        (arrival_auto_df["Reg"] == man_record["Reg"]) &
        (abs(arrival_auto_df["actual_arrival_time"] - man_record["actual_arrival_time"]) <= pd.Timedelta(hours=1))
    ]

In [98]:
auto_record = arrival_auto_df[arrival_auto_df["FID"]==125796].iloc[0]
arrival_man_df_prepared_deduplicated[
        # (arrival_auto_df['AirlineIATA'] == man_record['AirlineIATA']) &
        # (arrival_auto_df['FlightNumber'] == man_record['FlightNumber']) &
        # (arrival_auto_df['Bay'] == man_record['Bay']) &
        (arrival_man_df_prepared_deduplicated['STA_date'] == auto_record['FDate_date']) &
        (arrival_man_df_prepared_deduplicated["Reg"] == auto_record["Reg"]) &
        (abs(arrival_man_df_prepared_deduplicated["actual_arrival_time"] - auto_record["actual_arrival_time"]) <= pd.Timedelta(hours=1))
    ]

,FID,Airline,FlightNumber,Bay,STA,arrtime,statusarr,notearr,respon,full,...,Cargo,Reg,type,TGno,LAno,FSno,logtime,AirlineIATA,STA_date,actual_arrival_time
731,106281,CI,127,30,2025-07-23 00:00:00,2025-07-23 21:20:00,0,0.0,NaN,0,...,251,B18101,NaN,0.0,0.0,0.0,2025-07-23 19:27:38,CI,2025-07-23,2025-07-23 21:00:00
733,106283,CI,177,31,2025-07-23 00:00:00,2025-07-23 22:20:00,0,0.0,NaN,0,...,4080,B18101,NaN,0.0,0.0,0.0,2025-07-23 20:13:26,CI,2025-07-23,2025-07-23 22:00:00


Now given that we have the FID of arrival_man, we can restore data from them

In [101]:
# Correct way to create a mask for pandas DataFrame
arrival_man_mask = arrival_man_df_prepared_deduplicated["FID"].isin(exact_match)
exact_match_arrival_man = arrival_man_df_prepared_deduplicated[arrival_man_mask]
len(exact_match_arrival_man), len(arrival_auto_df)

(21298, 29046)

In [129]:
from datetime import date
time = date(2025,7,30)
exact_match_cargo = exact_match_arrival_man[exact_match_arrival_man["STA_date"]==time]["Cargo"].sum()
arrival_man_cargo = arrival_man_df_prepared_deduplicated[arrival_man_df_prepared_deduplicated["STA_date"]==time]["Cargo"].sum()
exact_match_cargo, arrival_man_cargo

(16374, 21762)

## Departure

In [149]:
# Step 2: Load data using pandas
# Load from SQLite (current state - arrival_auto)
conn = sqlite3.connect(SQL_DATABASE_PATH)
departure_sql = pd.read_sql("SELECT * FROM departures", conn)

# Load from CSV (source of truth - arrival_man)
departure_man_df = pd.read_csv("database/depart_man_bk.csv")

print(f"Arrival Auto (SQLite) rows: {len(departure_sql)}")
print(f"Arrival Man (CSV) rows: {len(departure_man_df)}")
print(f"\nArrival Auto columns: {list(departure_sql.columns)}")
print(f"Arrival Man columns (first 20): {list(departure_man_df.columns[:20])}")

departure_auto_df = departure_sql.copy()

Arrival Auto (SQLite) rows: 29026
Arrival Man (CSV) rows: 31847

Arrival Auto columns: ['FID', 'AircraftType', 'AirlineIATA', 'ArrivalAirportIATA', 'Bay', 'Cancel', 'Cargo', 'DepartureDateTime', 'FDate', 'FlightNumber', 'LoadCapacity', 'Passanger', 'RWYDEP', 'Reg', 'SCHE_TYPE', 'SeatCapacity', 'amhsATD', 'logtime']
Arrival Man columns (first 20): ['FID', 'Airline', 'FlightNumber', 'Bay', 'ckinisland', 'ckindeskstart', 'ckindeskstart2', 'ckindeskend2', 'ckindeskend', 'STD', 'boardtime', 'deptime', 'statusdep', 'notedep', 'respon', 'full', 'Passanger', 'Infant', 'transit', 'transit_na']


/tmp/ipykernel_2788176/1811356492.py:7: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  departure_man_df = pd.read_csv("database/depart_man_bk.csv")


In [150]:
# Step 3: Prepare data for matching using composite key

# The matching keys are:
# - AirlineIATA (auto) <--> Airline (man)
# - Bay (auto) <--> Bay (man)
# - FlightNumber (auto) <--> FlightNumber (man)
# - FDate (auto) <--> STD (%Y-%m-%d only) (man)
# - Reg (auto) <--> Reg (man)
# - amhsATD (%Y-%m-%d-h) <--> deptime (%Y-%m-%d-%h) (man)

# Prepare departure_man_df

# Rename Airline to AirlineIATA in man data for easier merging
departure_man_df_prepared = departure_man_df.copy()
departure_man_df_prepared['AirlineIATA'] = departure_man_df_prepared['Airline']

# Convert STA to date format to match FDate
departure_man_df_prepared['STD_date'] = pd.to_datetime(departure_man_df_prepared['STD'], errors='coerce').dt.date
departure_auto_df['FDate_date'] = pd.to_datetime(departure_auto_df['FDate'], errors='coerce').dt.date

# int the BAY column for both of the dataframe
departure_man_df_prepared['Bay'] = pd.to_numeric(departure_man_df['Bay'], errors='coerce').fillna(0).astype(int)
departure_auto_df['Bay'] = pd.to_numeric(departure_auto_df['Bay'], errors='coerce').fillna(0).astype(int)

# int the FlightNumber
departure_man_df_prepared['FlightNumber'] = pd.to_numeric(departure_man_df_prepared['FlightNumber'], errors='coerce')
departure_man_df_prepared = departure_man_df_prepared.dropna(subset=['FlightNumber'])  # Drop rows where FlightNumber is NaN
departure_man_df_prepared['FlightNumber'] = departure_man_df_prepared['FlightNumber'].astype(int)

departure_auto_df['FlightNumber'] = pd.to_numeric(departure_auto_df['FlightNumber'], errors='coerce')
departure_auto_df = departure_auto_df.dropna(subset=['FlightNumber'])
departure_auto_df['FlightNumber'] = departure_auto_df['FlightNumber'].astype(int)

# actual departure time
# amhsATA (%Y-%m-%d-h) <--> arrtime (%Y-%m-%d-%h) (man)
departure_man_df_prepared['actual_departure_time'] = pd.to_datetime(departure_man_df['deptime'], errors='coerce').dt.floor('h')
departure_auto_df['actual_departure_time'] = pd.to_datetime(departure_auto_df['amhsATD'], errors='coerce').dt.floor('h')


In [118]:
def deduplicate_manual_records(df):
    """Remove duplicate manual records, keeping only the first instance for each truly identical record"""
    
    # Create a composite key using ALL data fields (except FID which is unique)
    # Convert all columns to string and concatenate
    df_copy = df.copy()
    
    # Get all columns except FID
    columns_to_compare = [col for col in df_copy.columns if col != 'FID']
    
    # Create composite key from all fields
    df_copy['composite_key'] = df_copy[columns_to_compare].astype(str).agg('_'.join, axis=1)
    
    # Keep only the first occurrence of each truly identical record
    deduplicated = df_copy.drop_duplicates(subset=['composite_key'], keep='first')
    
    # Remove the temporary composite key column
    deduplicated = deduplicated.drop('composite_key', axis=1)
    
    return deduplicated

# Apply deduplication
departure_man_df_prepared_deduplicated = deduplicate_manual_records(departure_man_df_prepared)

print(f"Original manual records: {len(departure_man_df_prepared)}")
print(f"After deduplication: {len(departure_man_df_prepared_deduplicated)}")
print(f"Removed {len(departure_man_df_prepared) - len(departure_man_df_prepared_deduplicated)} truly identical records")

Original manual records: 31109
After deduplication: 29165
Removed 1944 truly identical records


In [123]:
from tqdm import tqdm

# Step 3.5: Test the composite key matching
# Iterate through the arrival_man to check each man
no_match = []
more_than_one_match = []
exact_match = []
exact_auto_match_set = set()
auto_one_to_many_fid = []


for i in tqdm(range(len(departure_man_df_prepared_deduplicated))):
    man_record = departure_man_df_prepared_deduplicated.iloc[i]
    
    # man --> auto
    auto_match = departure_auto_df[
        # (departure_auto_df['AirlineIATA'] == man_record['AirlineIATA']) &
        # (departure_auto_df['FlightNumber'] == man_record['FlightNumber']) &
        # (departure_auto_df['Bay'] == man_record['Bay']) &
        (departure_auto_df['FDate_date'] == man_record['STD_date']) &
        (departure_auto_df["Reg"] == man_record["Reg"]) &
        (abs(departure_auto_df["acutal_departure_time"] - man_record["actual_departure_time"]) <= pd.Timedelta(hours=1))
    ]
    if not auto_match.empty:
        # Only update if len(match) == 1
        if len(auto_match) == 1:
            exact_match.append(man_record["FID"])
            if auto_match.iloc[0]["FID"] in exact_auto_match_set:
                auto_one_to_many_fid.append(auto_match.iloc[0]["FID"])
            else:
                exact_auto_match_set.add(auto_match.iloc[0]["FID"])

            # Check if this match has already been matched by other man_record

        else: # len(match) is not 1, needs to be inspected
            more_than_one_match.append(man_record["FID"])
    else: # No match len(match) == 0
        no_match.append(man_record["FID"])

len(no_match), len(more_than_one_match), len(exact_match), len(auto_one_to_many_fid)

100%|██████████| 29165/29165 [01:49<00:00, 265.90it/s]


(6138, 9, 23018, 30)

Now given that we have the FID of arrival_man, we can restore data from them

In [126]:
# Correct way to create a mask for pandas DataFrame
departure_man_mask = departure_man_df_prepared_deduplicated["FID"].isin(exact_match)
exact_match_departure_man = departure_man_df_prepared_deduplicated[departure_man_mask]

In [128]:
from datetime import date
time = date(2025,7,30)
exact_match_cargo = exact_match_departure_man[exact_match_departure_man["STD_date"]==time]["Cargo"].sum()
departure_man_cargo = departure_man_df_prepared_deduplicated[departure_man_df_prepared_deduplicated["STD_date"]==time]["Cargo"].sum()
exact_match_cargo, departure_man_cargo

(33108, 33325)

## Step 4: Perform Restoration Logic

Define your restoration logic here. Common scenarios:
- Fill missing `Passanger`
- Fill `cargo`



### Arrival

In [151]:
arrival_modified_sql = arrival_sql.copy()

# Revert back to SQL Datframe
conn = sqlite3.connect(SQL_DATABASE_PATH)
arrival_sql = pd.read_sql("SELECT * FROM arrivals", conn)

In [158]:
# Get the one to one match of arrival_auto data
from tqdm import tqdm

# Step 3.5: Test the composite key matching
# Iterate through the arrival_man to check each man
no_match = []
more_than_one_match = []
exact_match = []
exact_auto_match_set = set()
auto_one_to_many_fid = []

for i in tqdm(range(len(exact_match_arrival_man))):
    man_record = exact_match_arrival_man.iloc[i]
    
    # man --> auto
    auto_match = arrival_auto_df[
        (arrival_auto_df['FDate_date'] == man_record['STA_date']) &
        (arrival_auto_df["Reg"] == man_record["Reg"]) &
        (abs(arrival_auto_df["actual_arrival_time"] - man_record["actual_arrival_time"]) <= pd.Timedelta(hours=1))
    ]
    if not auto_match.empty:
        if len(auto_match) == 1:
            # Get the auto FID
            auto_fid = auto_match.iloc[0]["FID"]
            
            # Perform restoring logic
            arrival_modified_sql.loc[arrival_modified_sql["FID"] == auto_fid, "Passanger"] = man_record["Passanger"]
            arrival_modified_sql.loc[arrival_modified_sql["FID"] == auto_fid, "Cargo"] = man_record["Cargo"]
            
            # Track the match
            exact_match.append(man_record["FID"])
            exact_auto_match_set.add(auto_fid)
            
        else:  # len(match) is not 1
            more_than_one_match.append(man_record["FID"])
            auto_one_to_many_fid.extend(auto_match["FID"].tolist())
            
    else:  # No match
        no_match.append(man_record["FID"])

print(f"Exact matches: {len(exact_match)}")
print(f"More than one match: {len(more_than_one_match)}")
print(f"No match: {len(no_match)}")
print(f"Unique auto FIDs matched: {len(exact_auto_match_set)}")

100%|██████████| 21298/21298 [01:43<00:00, 206.07it/s]

Exact matches: 21298
More than one match: 0
No match: 0
Unique auto FIDs matched: 21269


### Departure

In [159]:
departure_modified_sql = departure_sql.copy()

# Revert back to original SQL dataframe 
conn = sqlite3.connect(SQL_DATABASE_PATH)
departure_sql = pd.read_sql("SELECT * FROM departures", conn)

In [160]:
# Get the one to one match of departure_auto data
from tqdm import tqdm

# Step 3.5: Test the composite key matching for departures
# Iterate through the departure_man to check each man
no_match_departure = []
more_than_one_match_departure = []
exact_match_departure = []
exact_auto_match_set_departure = set()
auto_one_to_many_fid_departure = []

for i in tqdm(range(len(exact_match_departure_man))):
    man_record = exact_match_departure_man.iloc[i]
    
    # man --> auto
    auto_match = departure_auto_df[
        (departure_auto_df['FDate_date'] == man_record['STD_date']) &
        (departure_auto_df["Reg"] == man_record["Reg"]) &
        (abs(departure_auto_df["actual_departure_time"] - man_record["actual_departure_time"]) <= pd.Timedelta(hours=1))
    ]
    
    if not auto_match.empty:
        if len(auto_match) == 1:
            # Get the auto FID
            auto_fid = auto_match.iloc[0]["FID"]
            
            # Perform restoring logic
            departure_modified_sql.loc[departure_modified_sql["FID"] == auto_fid, "Passanger"] = man_record["Passanger"]
            departure_modified_sql.loc[departure_modified_sql["FID"] == auto_fid, "Cargo"] = man_record["Cargo"]
            
            # Track the match
            exact_match_departure.append(man_record["FID"])
            exact_auto_match_set_departure.add(auto_fid)
            
        else:  # len(match) is not 1
            more_than_one_match_departure.append(man_record["FID"])
            auto_one_to_many_fid_departure.extend(auto_match["FID"].tolist())
            
    else:  # No match
        no_match_departure.append(man_record["FID"])

print(f"Departure Results:")
print(f"Exact matches: {len(exact_match_departure)}")
print(f"More than one match: {len(more_than_one_match_departure)}")
print(f"No match: {len(no_match_departure)}")
print(f"Unique auto FIDs matched: {len(exact_auto_match_set_departure)}")

100%|██████████| 23018/23018 [01:50<00:00, 207.47it/s]

Departure Results:
Exact matches: 23018
More than one match: 0
No match: 0
Unique auto FIDs matched: 22988


At this point both `arrival_modified_sql` and `departure_modified_sql` should have the dataframe read from the sql database and the modified passanger and cargo. Can be checked as following
- Unmodified Dataframe has 70% Null for Passanger Field
- Unmodified Dataframe has all 0s in Cargo

In [179]:
# Check the passanger NULL
arrival_passanger_null_perct = arrival_modified_sql["Passanger"].isna().sum()/len(arrival_modified_sql)
departure_passanger_null_perct = departure_modified_sql["Passanger"].isna().sum()/len(departure_modified_sql)

print(f"Arrival Passanger NA Perct: {arrival_passanger_null_perct}\nDeparture Passanger NA Perct: {departure_passanger_null_perct}")

# Check the Cargo unique values
arrival_cargo_unique = len(arrival_modified_sql["Cargo"].unique())
departure_cargo_unique = len(departure_modified_sql["Cargo"].unique())
print(f"Arrival Cargo unique count: {arrival_cargo_unique}")
print(f"Departure Cargo unique count: {departure_cargo_unique}")

arrival_auto_modified = arrival_auto_df.copy()
departure_auto_modified = departure_auto_df.copy()

Arrival Passanger NA Perct: 0.2425815768662105
Departure Passanger NA Perct: 0.1908978157513953
Arrival Cargo unique count: 2002
Departure Cargo unique count: 2214


Now check the original and modified version are still the same dataframe besides the passanger and cargo fields

In [162]:
# Compare arrival_modified_sql with arrival_sql
print("=== ARRIVAL COMPARISON ===")

# Check if all FIDs match
arrival_fid_match = set(arrival_modified_sql['FID']) == set(arrival_sql['FID'])
print(f"FID sets match: {arrival_fid_match}")

# Check if all columns match
arrival_columns_match = list(arrival_modified_sql.columns) == list(arrival_sql.columns)
print(f"Column sets match: {arrival_columns_match}")

# Compare each column (excluding Passanger and Cargo)
columns_to_compare = [col for col in arrival_sql.columns if col not in ['Passanger', 'Cargo']]
arrival_differences = {}

for col in columns_to_compare:
    if col in arrival_modified_sql.columns:
        # Check if values are equal
        values_equal = arrival_modified_sql[col].equals(arrival_sql[col])
        arrival_differences[col] = values_equal
        
        if not values_equal:
            # Find specific differences
            diff_mask = arrival_modified_sql[col] != arrival_sql[col]
            diff_count = diff_mask.sum()
            print(f"❌ {col}: {diff_count} differences found")
        else:
            print(f"✅ {col}: No differences")
    else:
        print(f"⚠️ {col}: Column missing in modified data")

# Check Passanger and Cargo specifically
print(f"\n=== PASSANGER & CARGO CHANGES ===")
passanger_changed = not arrival_modified_sql['Passanger'].equals(arrival_sql['Passanger'])
cargo_changed = not arrival_modified_sql['Cargo'].equals(arrival_sql['Cargo'])

print(f"Passanger values changed: {passanger_changed}")
if passanger_changed:
    passanger_diff_count = (arrival_modified_sql['Passanger'] != arrival_sql['Passanger']).sum()
    print(f"  - {passanger_diff_count} Passanger values modified")

print(f"Cargo values changed: {cargo_changed}")
if cargo_changed:
    cargo_diff_count = (arrival_modified_sql['Cargo'] != arrival_sql['Cargo']).sum()
    print(f"  - {cargo_diff_count} Cargo values modified")

# Summary
other_columns_changed = any(not status for status in arrival_differences.values())
print(f"\n=== SUMMARY ===")
print(f"Other columns unchanged: {not other_columns_changed}")
print(f"Only Passanger and Cargo modified: {not other_columns_changed and (passanger_changed or cargo_changed)}")

=== ARRIVAL COMPARISON ===
FID sets match: True
Column sets match: True
✅ FID: No differences
✅ AircraftType: No differences
✅ AirlineIATA: No differences
✅ ArrivalDateTime: No differences
✅ Bay: No differences
✅ Cancel: No differences
✅ DepartureAirportIATA: No differences
✅ FDate: No differences
✅ FlightNumber: No differences
✅ LoadCapacity: No differences
✅ RWYARR: No differences
✅ Reg: No differences
✅ SCHE_TYPE: No differences
✅ SeatCapacity: No differences
✅ amhsATA: No differences
✅ logtime: No differences

=== PASSANGER & CARGO CHANGES ===
Passanger values changed: True
  - 27359 Passanger values modified
Cargo values changed: True
  - 9432 Cargo values modified

=== SUMMARY ===
Other columns unchanged: True
Only Passanger and Cargo modified: True


In [163]:
# Compare departure_modified_sql with departure_sql
print("\n=== DEPARTURE COMPARISON ===")

# Check if all FIDs match
departure_fid_match = set(departure_modified_sql['FID']) == set(departure_sql['FID'])
print(f"FID sets match: {departure_fid_match}")

# Check if all columns match
departure_columns_match = list(departure_modified_sql.columns) == list(departure_sql.columns)
print(f"Column sets match: {departure_columns_match}")

# Compare each column (excluding Passanger and Cargo)
columns_to_compare = [col for col in departure_sql.columns if col not in ['Passanger', 'Cargo']]
departure_differences = {}

for col in columns_to_compare:
    if col in departure_modified_sql.columns:
        # Check if values are equal
        values_equal = departure_modified_sql[col].equals(departure_sql[col])
        departure_differences[col] = values_equal
        
        if not values_equal:
            # Find specific differences
            diff_mask = departure_modified_sql[col] != departure_sql[col]
            diff_count = diff_mask.sum()
            print(f"❌ {col}: {diff_count} differences found")
        else:
            print(f"✅ {col}: No differences")
    else:
        print(f"⚠️ {col}: Column missing in modified data")

# Check Passanger and Cargo specifically
print(f"\n=== PASSANGER & CARGO CHANGES ===")
passanger_changed = not departure_modified_sql['Passanger'].equals(departure_sql['Passanger'])
cargo_changed = not departure_modified_sql['Cargo'].equals(departure_sql['Cargo'])

print(f"Passanger values changed: {passanger_changed}")
if passanger_changed:
    passanger_diff_count = (departure_modified_sql['Passanger'] != departure_sql['Passanger']).sum()
    print(f"  - {passanger_diff_count} Passanger values modified")

print(f"Cargo values changed: {cargo_changed}")
if cargo_changed:
    cargo_diff_count = (departure_modified_sql['Cargo'] != departure_sql['Cargo']).sum()
    print(f"  - {cargo_diff_count} Cargo values modified")

# Summary
other_columns_changed = any(not status for status in departure_differences.values())
print(f"\n=== SUMMARY ===")
print(f"Other columns unchanged: {not other_columns_changed}")
print(f"Only Passanger and Cargo modified: {not other_columns_changed and (passanger_changed or cargo_changed)}")


=== DEPARTURE COMPARISON ===
FID sets match: True
Column sets match: True
✅ FID: No differences
✅ AircraftType: No differences
✅ AirlineIATA: No differences
✅ ArrivalAirportIATA: No differences
✅ Bay: No differences
✅ Cancel: No differences
✅ DepartureDateTime: No differences
✅ FDate: No differences
✅ FlightNumber: No differences
✅ LoadCapacity: No differences
✅ RWYDEP: No differences
✅ Reg: No differences
✅ SCHE_TYPE: No differences
✅ SeatCapacity: No differences
✅ amhsATD: No differences
✅ logtime: No differences

=== PASSANGER & CARGO CHANGES ===
Passanger values changed: True
  - 24706 Passanger values modified
Cargo values changed: True
  - 12484 Cargo values modified

=== SUMMARY ===
Other columns unchanged: True
Only Passanger and Cargo modified: True


## Step 5: Choose Your Update Strategy

**Option A: Replace entire table** (simpler but recreates indexes)
**Option B: Selective updates** (more efficient, preserves structure)

I recommend Option B for production use.



In [164]:
# OPTION B: Selective updates (RECOMMENDED)
# Only update rows that actually changed
cursor = conn.cursor()

# Prepare the update statement for fields you restored
update_fields = ['Passanger', 'Cargo']  # Only the fields that were modified
set_clause = ", ".join([f"{field} = ?" for field in update_fields])
update_query = f"UPDATE arrivals SET {set_clause} WHERE FID = ?"

# Track updates
updates_performed = 0

# Update arrivals table
for idx, row in arrival_modified_sql.iterrows():
    # Only update if something actually changed
    original_row = arrival_sql[arrival_sql['FID'] == row['FID']]
    if not original_row.empty:
        changed = False
        for field in update_fields:
            # Check if the value actually changed (not just from NaN to value)
            original_value = original_row.iloc[0][field]
            new_value = row[field]
            
            # Consider it changed if:
            # 1. Original was NaN and new is not NaN, OR
            # 2. Both are not NaN but values are different
            if (pd.isna(original_value) and pd.notna(new_value)) or \
               (pd.notna(original_value) and pd.notna(new_value) and original_value != new_value):
                changed = True
                break
        
        if changed:
            values = [row[field] for field in update_fields] + [row['FID']]
            cursor.execute(update_query, values)
            updates_performed += 1

print(f"✅ Updated {updates_performed} arrival records in SQLite database")

# Now do the same for departures
update_query_departure = f"UPDATE departures SET {set_clause} WHERE FID = ?"
updates_performed_departure = 0

for idx, row in departure_modified_sql.iterrows():
    # Only update if something actually changed
    original_row = departure_sql[departure_sql['FID'] == row['FID']]
    if not original_row.empty:
        changed = False
        for field in update_fields:
            # Check if the value actually changed
            original_value = original_row.iloc[0][field]
            new_value = row[field]
            
            if (pd.isna(original_value) and pd.notna(new_value)) or \
               (pd.notna(original_value) and pd.notna(new_value) and original_value != new_value):
                changed = True
                break
        
        if changed:
            values = [row[field] for field in update_fields] + [row['FID']]
            cursor.execute(update_query_departure, values)
            updates_performed_departure += 1

conn.commit()
print(f"✅ Updated {updates_performed_departure} departure records in SQLite database")
print(f"📊 Total updates: {updates_performed + updates_performed_departure} records")

✅ Updated 20743 arrival records in SQLite database
✅ Updated 21396 departure records in SQLite database
📊 Total updates: 42139 records


In [178]:
# Check the type of Cargo
conn = sqlite3.connect(SQL_DATABASE_PATH)
arrival_sql_modified = pd.read_sql("SELECT * FROM arrivals", conn)
arrival_sql_modified["Passanger"].head(20)

0       0.0
1       NaN
2       NaN
3     151.0
4     201.0
5     143.0
6       0.0
7     179.0
8     135.0
9     156.0
10      NaN
11      NaN
12      NaN
13    156.0
14      NaN
15    173.0
16    182.0
17    167.0
18      NaN
19      NaN
Name: Passanger, dtype: float64